# Explore the dbstim API

**Connect → inspect metrics → create an experiment → download → check out a session → record a run → sync.**

This walkthrough uses [dbstim 1.0](https://github.com/StanfordDigitalBrainProject/dbstim) (release v1.0.0), the Python client for [dbstimuli.com](https://dbstimuli.com). It follows dbstim's own [notebook](https://github.com/StanfordDigitalBrainProject/dbstim/blob/v1.0.0/notebooks/api.ipynb), which is the reference for the current API.

Two journeys, one after the other:

- **A. Download videos for your own code:** create an experiment from filters, a count and a seed, then download a portable BIDS stimuli folder.
- **B. Track a run:** check a subject session out to this computer, record a run offline, then sync it.

The website stores the catalog and experiments. dbstim fetches stimuli and records what your task showed; PGL or PsychoPy plays them. Run one cell at a time. Creating, scheduling, checkout and sync below change account data; reading metrics and opening an experiment do not.

## API objects at a glance

Main objects and selected methods, not every helper. Arrows show what each object creates, returns, or contains.

```mermaid
classDiagram
    direction TB

    class Client {
        metrics() list~str~
        create(name, filters, count, seed) Experiment
        open(name, revision) Experiment
        close()
    }
    class Filter {
        str metric
        str op
        float|bool value
    }
    class Experiment {
        str name
        int revision
        videos
        download(folder) Path
        schedule(subjects, sessions, runs, trials_per_run) Experiment
        checkout(subject, session, folder) Session
    }
    class Session {
        Path folder
        int subject
        int session
        tuple runs
        run(number) Run
        sync(client) dict
    }
    class Run {
        iterate() Trial
        started(trial, onset)
        completed(trial, duration, result)
        abandoned(trial)
    }
    class Trial {
        int number
        Path file
        str video_id
        str role
    }

    Client ..> Experiment : create / open
    Experiment --> Filter : zero or more
    Experiment ..> Experiment : schedule saves a new revision
    Experiment ..> Session : checkout reserves and downloads
    Session ..> Run : run(n) opens the journal
    Run --> Trial : trials still to show
```

**Boundary:** `connect()`, `create()`, `open()`, `download()`, `schedule()`, `checkout()` and `sync()` talk to the website. `dbstim.session(folder)` and everything inside `with session.run(n)` work offline; each call is written to disk before it returns. PGL/PsychoPy handles playback; dbstim does not.

If your notebook viewer shows the Mermaid source instead of a figure, use a Mermaid-capable Markdown renderer.

## 1. Setup

Use Python 3.11–3.12. From this directory:

```sh
uv sync
uv run jupyter lab api-walkthrough.ipynb
```

In VS Code, select `notebooks/.venv/bin/python`. `uv sync` installs dbstim v1.0.0. Sign in with your dbstimuli.com account. Set the `DBSTIM_URL` environment variable to try a local test server instead of dbstimuli.com.

In [ ]:
import sys

print("Kernel Python:", sys.executable)
assert sys.version_info >= (3, 11), "Select a Python 3.11+ kernel."


### Imports
Run from here after restarting the kernel.

In [ ]:
import os
import time

from IPython.display import Video as VideoPreview, display

import dbstim
from dbstim import Filter

print("dbstim", dbstim.__version__)


## 2. Connect and inspect the metrics

`connect()` asks for your username and password; nothing is saved, so never type the password into a cell. `metrics()` lists the metric names you can filter on. Missing measurements are not zero or absence.

In [ ]:
website_url = os.environ.get("DBSTIM_URL", "https://dbstimuli.com")

client = dbstim.connect(url=website_url)
print("Connected to", website_url)
client.metrics()


## A. Download videos for your own code

### 3. Create an experiment

`create()` saves a frozen selection: every video matching all filters, sampled by `count` and `seed`. Filters are combined with AND. Names are unique in your account, so running the next cell again with the same name fails.

To reopen a saved experiment instead, use `client.open(NAME)` for the latest revision or `client.open(NAME, revision=1)` for an earlier one. Each revision needs its own download folder.

In [ ]:
NAME = "API walkthrough"
filters = [Filter("duration_seconds", ">=", 10)]
experiment = client.create(NAME, filters=filters, count=4, seed=1)
experiment.name, experiment.revision, experiment.videos


### 4. Download and preview

`download()` writes the videos under `stimuli/` with `manifest.json` (revision, filters, sizes and SHA-256) and `manifest.csv` (filename, index, id, ... for PGL). The folder works without dbstim; no subjects, scheduling or tracking are involved. This cell transfers media only when you run it.

In [ ]:
folder = experiment.download("api_walkthrough_bids")
sorted(path.relative_to(folder).as_posix() for path in folder.rglob("*") if path.is_file())


In [ ]:
print((folder / "stimuli" / "manifest.csv").read_text())


In [ ]:
first_video = sorted((folder / "stimuli").glob("*.mp4"))[0]
display(VideoPreview(filename=str(first_video), embed=True, width=480))


## B. Track a run

### 5. Schedule and check out a session (online)

`schedule()` saves a new revision with subjects, sessions and runs. `checkout()` reserves one subject session for this computer and downloads only its videos. Another computer would need `takeover=True`; runs the first computer syncs afterwards are kept as flagged attempts.

In [ ]:
scheduled = experiment.schedule(subjects=1, runs=2, trials_per_run=2)
checked = scheduled.checkout(subject=1, session=1, folder="api_walkthrough_sub01_ses01")
checked.subject, checked.session, checked.runs


### 6. Record a run (offline)

`dbstim.session(folder)` reopens the checkout without the network. Inside `with session.run(n)`, iterating gives the trials that have no outcome yet, in order, and each call is written to disk before it returns. Previewing videos above does not record anything.

A real task plays `trial.file` with PGL; see dbstim's [`examples/pgl_tracked_movie_task.py`](https://github.com/StanfordDigitalBrainProject/dbstim/blob/v1.0.0/examples/pgl_tracked_movie_task.py). Here, stand-in frame times feed `frame_summary()`, whose output goes into the result. Report `run.abandoned(trial)` instead of `completed` for a trial that was not shown to the end.

In [ ]:
session = dbstim.session("api_walkthrough_sub01_ses01")
with session.run(1) as run:
    t0 = time.monotonic()
    for trial in run:
        run.started(trial, onset=time.monotonic() - t0)
        frames = [i / 30 for i in range(30)]  # stand-in for PGL's movieTimes
        evidence = dbstim.frame_summary([t0 + t for t in frames], frames, 1.0)
        run.completed(trial, duration=evidence["seconds_shown"], result=evidence)


Leaving the block wrote a BIDS events file and sidecar from the journal. If the program had crashed, reopening the run would record the open trial as `abandoned` and continue with the next one; nothing is shown twice.

In [ ]:
events = next(session.folder.glob("sub-*/ses-*/beh/*_run-01_events.tsv"))
print(events.read_text())


### 7. Sync (online, afterwards)

`sync()` uploads each finished run once and returns `{run number: outcome}`: `accepted`, `already synced`, `kept as flagged attempt`, or `waiting` for a run not finished yet. Run 2 has not been recorded, so it waits. If the server refuses a run, the other runs still sync and `SyncError` lists the refused ones. Keep the session folder until sync is confirmed.

In [ ]:
session.sync(client=client)


## 8. Finish

Closing the client forgets local credentials; it does not delete saved experiments or local files. Clear notebook outputs before sharing: they may contain account data, experiment names and embedded videos.

In [ ]:
client.close()
